# Fase 4: Análisis Diferencial, Perfilado Agronómico y Validación

**Project Terra** — Caracterización de ecorregiones funcionales y validación cruzada.

En este notebook:
1. **Perfil Edafoclimático:** Analizamos medias, medianas y boxplots paralelos por clúster.
2. **Firma Multivariada (Radar Chart):** Visualizamos las identidades agronómicas de cada grupo.
3. **Análisis de Tipos de Suelo (`Soil_Type`):** Evaluamos si las ecorregiones reflejan afinidad por sustratos específicos (Arcilla, Limo, Salino, etc.).
4. **Validación Agronómica vs. `Crop`:** Analizamos la distribución de cultivos en cada clúster, calculamos el **Purity Score** y la significancia estadística con **Kruskal-Wallis**.
5. **Conclusiones Estratégicas:** Resumen agronómico para la toma de decisiones y asignación de recursos agrícolas.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.profiling import create_radar_chart, contingency_table, purity_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Carga de Datos Clustered
Cargamos `sensor_Crop_Dataset_clustered.csv` generado en la fase anterior.


In [ ]:
data_path = ROOT_DIR / 'data' / 'processed' / 'sensor_Crop_Dataset_clustered.csv'
df = pd.read_csv(data_path)
print(f'Muestras: {len(df):,} | Clústeres: {df["kmeans_cluster"].nunique()}')
df.head()


## 2. Perfil Ambiental por Ecorregión
Examinamos la media y desviación estándar de cada variable en las ecorregiones.


In [ ]:
feature_cols = ['Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Humidity', 'pH_Value', 'Rainfall']
means_by_cluster = df.groupby('kmeans_cluster')[feature_cols].mean()
display(means_by_cluster.round(2))

# Boxplots comparativos para cada variable agrupados por clúster
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(x='kmeans_cluster', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Distribución de {col} por Ecorregión', fontsize=12)
    axes[i].set_xlabel('Clúster K-Means')

fig.delaxes(axes[-1])  # Eliminar último subplot vacío
plt.tight_layout()
plt.show()


## 3. Firmas Agronómicas Multivariadas (Radar Charts)
El polígono de radar sintetiza la 'personalidad ambiental' de cada zona en un solo gráfico.


In [ ]:
fig_radar = create_radar_chart(df, cluster_col='kmeans_cluster', feature_cols=feature_cols)
fig_radar.show()


## 4. Análisis de Composición de Suelos (`Soil_Type`)
Evaluamos la distribución de tipos de suelo presentes en cada ecorregión.


In [ ]:
soil_ct = contingency_table(df, cluster_col='kmeans_cluster', label_col='Soil_Type', normalize='index')
print('Porcentaje de tipos de suelo dentro de cada clúster:')
display(soil_ct)

# Gráfico de barras apiladas
soil_counts = pd.crosstab(df['kmeans_cluster'], df['Soil_Type'], normalize='index') * 100
soil_counts.plot(kind='bar', stacked=True, colormap='Spectral', figsize=(10, 6))
plt.title('Composición de Tipos de Suelo por Ecorregión (%)')
plt.xlabel('Ecorregión')
plt.ylabel('Porcentaje')
plt.legend(title='Tipo de Suelo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 5. Validación Agronómica contra `Crop` (Cultivo Real)
Contrastamos las ecorregiones descubiertas con los cultivos reales reportados.
Calculamos la pureza del agrupamiento y la matriz de afinidad de cultivo por ecorregión.


In [ ]:
crop_ct = contingency_table(df, cluster_col='kmeans_cluster', label_col='Crop', normalize='index')
print('Distribución porcentual de cultivos por clúster:')
display(crop_ct)

purity = purity_score(df['Crop'], df['kmeans_cluster'])
print(f'\nPureza global de las ecorregiones respecto al cultivo: {purity*100:.2f}%')

# Heatmap de afinidad clúster vs cultivo
plt.figure(figsize=(14, 7))
sns.heatmap(crop_ct, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': '% dentro del clúster'})
plt.title('Mapa de Calor de Afinidad: Ecorregión vs. Cultivo Real (%)')
plt.xlabel('Cultivo')
plt.ylabel('Ecorregión')
plt.show()


## 6. Pruebas de Significancia Estadística (Kruskal-Wallis)
Comprobamos formalmente si las diferencias observadas entre ecorregiones para cada variable ambiental son estadísticamente significativas.


In [ ]:
kw_results = []
for col in feature_cols:
    groups = [group[col].values for _, group in df.groupby('kmeans_cluster')]
    stat, p_val = stats.kruskal(*groups)
    kw_results.append({
        'Variable': col,
        'Kruskal-Wallis H': stat,
        'p-value': p_val,
        'Diferencia Significativa (p < 0.01)': p_val < 0.01
    })

kw_df = pd.DataFrame(kw_results)
display(kw_df)


## 7. Conclusiones y Caracterización de las Ecorregiones

A partir de los perfiles analizados, cada clúster representa una zona ecológica funcional:

1. **Ecorregión 0 (Bajo Nitrógeno, Clima Templado):** Nicho ideal para cultivos de secano o rotación con leguminosas que fijen nitrógeno.
2. **Ecorregión 1 (Alta Pluviosidad y Humedad Relativa):** Zona con abundante recurso hídrico, idónea para cultivos de alta demanda como Arroz o Caña.
3. **Ecorregión 2 (Rica en Fósforo y Potasio):** Suelos con alto contenido mineral, propicios para tubérculos y frutales.
4. **Ecorregión 3 (Temperatura Elevada y Clima Seco):** Demanda manejo de riego tecnificado y selección de variedades tolerantes al estrés térmico.
5. **Ecorregión 4 (Suelos Equilibrados / Neutros):** Zonas versátiles de alta productividad para cereales y hortalizas.

Este perfilado constituye la base del recomendador implementado en la aplicación de Streamlit (`app/main.py`).
